# Phase 3h: the call-only figures in the UBS reports of 2025 and 2026

Plan: `plans/phase_3h.md` (revision and amendments of 2026-09-26). Computation: `analysis/report_text.py` (shared text preparation), `analysis/phase3h_text.py` (export, manifest), `analysis/phase3h_terms.py` (terms, check files, deviation list), `analysis/phase3h_search.py` and `analysis/phase3h_read.py` (classes, outcomes, verified quotes); outputs in `data/phase3h/`. Descriptive: no test, no register entry.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase3h"
pd.set_option("display.max_colwidth", 200)

## Reports and manifest

In [ ]:
m = pd.read_csv(OUT / "manifest.csv", keep_default_na=False)
m[["file", "accession", "download_date", "publication_date", "pages", "ix_header_chars", "sec_script_tags_removed",
   "lines_differing_from_phase3b"]]

The manifest is complete for all 15 files; the three Inline XBRL files carry a hidden header of 1.8 to 4.7 million characters, removed before extraction. Fourteen files contain the script tag that EDGAR inserts on delivery, so `sha256_normalized` is the hash to compare with a new download.

## Deviations between Roman's tool and the pipeline

In [ ]:
d = pd.read_csv(OUT / "deviations.csv", keep_default_na=False)
d.groupby("cause").agg(pairs=("term", "size"), pipeline=("pipeline", "sum"), tool_raw=("tool_raw", "sum"))

All 71 deviations are explained before reading: 53 come from multi-line table headers that the export interleaves row by row, 17 from the distance between an outcome word and its keyword, and one from a hyphen repair. None of the hits that only the tool finds contains a forward-looking statement on one of the units; they are glossary entries, note headings, chart labels and an outlook sentence on Group RWA model updates that the pipeline also reads.

## (a) Does a call figure appear in the reports of 2025 and 2026?

In [ ]:
u = pd.read_csv(OUT / "units_3h.csv", keep_default_na=False)
u[["unit", "main_finding", "class_a_overall", "first_A_or_B", "comment_a"]]

In [ ]:
ca = pd.read_csv(OUT / "classes_a.csv", keep_default_na=False)
ca.pivot_table(index="unit", columns="report", values="class", aggfunc="first")

None of the ten call figures appears in a forward-looking sentence of the 13 reports of 2025 and 2026 (no class A, and no retrospective mention of a call figure). Five units are mentioned without their figure or with another one: the NCL run-down (T16, T25, T27) through new credit and market risk RWA ambitions of below USD 8bn and around USD 4bn, the phase-in of the higher capital requirements (T32) and a revised integration-cost total of around USD 15bn (T30); T33, T68, F11, F12 and F22 are not mentioned.

## (b) Outcome

In [ ]:
u[["unit", "deadline", "final_status", "actual_value", "last_mentioned_in"]]

In [ ]:
ob = pd.read_csv(OUT / "outcomes_b.csv", keep_default_na=False)
ob[ob.file.str.contains("_report")].pivot_table(index="unit", columns="report", values="status", aggfunc="first")

Two commitments are reported as met: NCL credit and market risk RWA fell to around USD 5bn at the end of 2025 and USD 4bn in 2026, far below the 40bn of the call, and the quarterly average LCR stayed between 177% and 183%, below 188%. Two were revised (integration costs to around USD 15bn, the progressive capital requirement to USD 6bn from USD 9bn), and the AT1 issuance of USD 3.5bn in 2024 exceeded the call's upper bound of USD 2bn.

Five commitments are no longer mentioned: NCL operational risk RWA, NCL capital release, the HoldCo reduction, the going concern capital ratio of around 18% and the Group LRD reduction. For F11 the reported ratio (19.0% at 30 June 2026) already exceeds the call figure, and for T25 the 2024 annual report, read here only for T27 and T33, states a capital release of over USD 6bn.

In [ ]:
u[["unit", "actual_value", "actual_quote", "actual_location", "comment_b"]]

## Quotes

In [ ]:
q = pd.read_csv(OUT / "quotes.csv", keep_default_na=False)
q.groupby("use").size()

Every quote in the tables above is checked verbatim (whitespace normalised) against the exported page text by `phase3h_read.py`, which refuses to write otherwise.

## Limits

One reader; C means not found with the listed terms. Several deadlines lie after 30 June 2026, and the NCL units are counted as B through the consistency rule, which treats the NCL run-down with another scope as the same reduction. Pillar 3 reports carry none of the commitments except the phase-in (T32) and one NCL sentence; the final status is therefore taken from the latest quarterly or annual report that states an outcome.